# 02 — 特征工程实验

本 Notebook 展示 70 维特征的提取过程：无量纲时域特征、频谱统计特征、动态谐波提取和跨通道比值。
与流水线一/二使用完全相同的 `feature_utils.py` 函数。

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.config import DATA_DIR, SKIPROWS, DATA_COLUMNS, SIGNAL_COLUMNS, FILE_ENCODING, SEPARATOR, WINDOW_SIZE, STEP_SIZE, SAMPLING_RATE
from src.feature_utils import (extract_time_domain_features, extract_freq_domain_features,
                                extract_features_for_window, FEATURE_COLUMNS, ALL_FEATURE_COLUMNS, N_FFT)
from scipy.fft import fft, fftfreq

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# 加载一个窗口的数据
import re
from src.config import LABEL_MAP

data_files = sorted(Path(DATA_DIR).glob('*.txt'))
fp = str([f for f in data_files if 'BF_10HZ' in f.name][0])

df = pd.read_csv(fp, skiprows=SKIPROWS, sep=SEPARATOR, header=None, names=DATA_COLUMNS, encoding=FILE_ENCODING, engine='python')
signal_data = df[SIGNAL_COLUMNS].values.astype(np.float64)

# 取第一个窗口
window = signal_data[:WINDOW_SIZE, :]
speed = 10  # Hz

print(f'窗口形状: {window.shape} (采样点 × 通道)')
print(f'转速: {speed} Hz, 采样率: {SAMPLING_RATE} Hz')

In [ ]:
# 1. 无量纲时域特征（X通道）
x_signal = window[:, 0]
td_features = extract_time_domain_features(x_signal)

print('X 通道 — 无量纲时域特征:')
print('=' * 50)
for name, val in td_features.items():
    bar = '█' * int(min(abs(val) * 12, 40))
    print(f'  {name:<25s} {val:10.4f}  {bar}')

print(f'\n  说明: 所有特征均通过比值运算消去了绝对幅值量级的影响，')
print(f'  对转速变化具有天然鲁棒性。峭度>3 表示冲击成分（轴承故障特征）。')

In [ ]:
# 2. 频域特征 + 动态谐波提取
fd_features = extract_freq_domain_features(x_signal, fs=SAMPLING_RATE, base_freq=speed, n_fft=N_FFT)

# 绘制频谱及谐波位置
spectrum = np.abs(fft(x_signal, n=N_FFT))[:N_FFT // 2]
freqs = fftfreq(N_FFT, 1.0 / SAMPLING_RATE)[:N_FFT // 2]

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(freqs, spectrum, color='#00e5ff', linewidth=0.8)
ax.set_xlim(0, 200)
ax.set_xlabel('频率 (Hz)', fontsize=12)
ax.set_ylabel('幅值', fontsize=12)
ax.set_title(f'X 通道 FFT 频谱 (n_fft={N_FFT}, 分辨率 {SAMPLING_RATE/N_FFT:.1f} Hz)', fontsize=13)
ax.grid(True, alpha=0.2)
ax.set_facecolor('#0a0e27')
fig.patch.set_facecolor('#0a0e27')

# 标注谐波位置
for h, color in [(1, '#ff9100'), (2, '#ff1744'), (3, '#e040fb')]:
    freq = speed * h
    ax.axvline(x=freq, color=color, linestyle='--', alpha=0.7, linewidth=1.2)
    ax.text(freq, ax.get_ylim()[1] * 0.85, f'{h}×fr ({freq} Hz)', color=color, fontsize=10, rotation=90)

plt.tight_layout()
plt.show()

print('\n频谱统计特征:')
for k, v in fd_features.items():
    print(f'  {k:<25s} {v:12.6f}')

In [ ]:
# 3. 完整特征提取（70 维）
all_features = extract_features_for_window(window, speed_hz=speed, fs=SAMPLING_RATE)

print(f'特征总数: {len(FEATURE_COLUMNS)} 维 (不含转速)')
print(f'含转速:   {len(ALL_FEATURE_COLUMNS)} 维\n')

# 按类别统计
categories = {
    '时域无量纲 (24维)': [c for c in FEATURE_COLUMNS if any(t in c for t in ['kurtosis','skewness','crest','shape','impulse','clearance'])],
    '频谱统计 (24维)': [c for c in FEATURE_COLUMNS if c.split('_')[-1].startswith('spec_')],
    '动态谐波 (12维)': [c for c in FEATURE_COLUMNS if 'harm_norm' in c],
    '跨通道比值 (9维)': [c for c in FEATURE_COLUMNS if c.startswith('cross_')],
}

for cat, cols in categories.items():
    vals = [all_features[c] for c in cols[:6]]  # 展示前6个
    print(f'{cat}:')
    for c, v in zip(cols[:6], vals):
        print(f'    {c:<35s} {v:10.4f}')
    if len(cols) > 6:
        print(f'    ... 共 {len(cols)} 维')
    print()

### 特征设计要点

1. **无量纲时域**：波峰因子/脉冲因子/裕度因子通过比值消除了振动幅值的绝对量级，使 5 Hz 和 30 Hz 的同故障窗口产生相似特征值
2. **动态谐波提取**：根据当前转速 fr 动态计算目标频率 (1×fr, 2×fr, 3×fr)，±3.5 Hz 搜索窗确保在 ~3.1 Hz 频率分辨率下准确捕获
3. **零填充 FFT**：8192 点 FFT 提供 ~3.1 Hz 分辨率，足以分辨 5 Hz 的转频
4. **跨通道比值**：捕获振动能量在各传感器间的分布模式，例如不对中故障的 X/Y 比值特征明显